# 14 — 2022 stress test

Measure how exceptional 2022 was relative to pre-closure distributions and decompose which physical-balance components moved.


In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from portugal_refining_resilience.config import get_paths
from portugal_refining_resilience.io import persist_dataframe, write_json

PATHS = get_paths(ROOT)
pd.set_option("display.max_columns", 100)


In [ ]:
panel = pd.read_csv(PATHS.processed / "fuel_annual_analytical_panel.csv")
records = []
for product, sub in panel.groupby("product"):
    pre = sub.loc[sub["year"].between(2005, 2020)]
    row2022 = sub.loc[sub["year"] == 2022]
    if row2022.empty:
        continue
    for outcome in ["exports_kt", "imports_kt", "demand_kt", "refinery_output_kt", "net_imports_kt", "net_import_dependence"]:
        if outcome not in sub or pre[outcome].dropna().shape[0] < 5:
            continue
        mean = pre[outcome].mean()
        sd = pre[outcome].std(ddof=1)
        value = float(row2022.iloc[0][outcome])
        z = (value - mean) / sd if sd and np.isfinite(sd) else np.nan
        records.append({"product": product, "outcome": outcome, "year": 2022, "value": value, "pre_2005_2020_mean": mean, "pre_2005_2020_sd": sd, "z_score": z})
stress = pd.DataFrame(records)
persist_dataframe(stress, PATHS.metrics / "stress_2022_metrics.csv")
display(stress.sort_values("z_score"))


In [ ]:
# Direct year-on-year movement around the stress year.
window = panel.loc[panel["year"].isin([2020, 2021, 2022, 2023])].copy()
cols = [c for c in ["exports_kt", "imports_kt", "demand_kt", "refinery_output_kt", "net_imports_kt"] if c in window.columns]
yoy_rows = []
for product, sub in window.groupby("product"):
    sub = sub.sort_values("year")
    for col in cols:
        changes = sub.set_index("year")[col].pct_change(fill_method=None) * 100
        for year, value in changes.dropna().items():
            yoy_rows.append({"product": product, "outcome": col, "year": int(year), "yoy_pct": value})
yoy = pd.DataFrame(yoy_rows)
persist_dataframe(yoy, PATHS.metrics / "stress_window_yoy_changes.csv")
display(yoy.loc[yoy["year"] == 2022])
